# CS781 Phase 3 — Rhetorical Role Prediction (Model 1: attention decoder, `ours_hybrid*`)
Starting point: our own Phase-1 attention-decoder legal LM (no external weights). Each sentence gets one of 13 rhetorical roles; CL and IT are trained jointly and evaluated separately. Experiments in this notebook:
1. **LoRMA+** (frozen backbone, multiplicative low-rank adaptation) with **context-aware** input (windows of consecutive sentences)
2. **LoRMA+ with sentence-only input** (ablation: does document context help?)
3. **Knowledge distillation** — 6-layer student (initialised from Phase-1) learns from the LoRMA+ teacher's sentence-level logits

Sentence representations are mean-pooled token states from a new bidirectional head layer, followed by a linear classifier. Each experiment writes a `prediction.json` (one label per sentence) and a zip.

## 1. Setup and imports

In [1]:
import os
import sys
import json
import glob
import time
import math
import pickle
import random
import zipfile
import subprocess

import numpy as np


def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args])


try:
    import regex
except ImportError:
    pip_install("regex")
    import regex

gpu_check = "import jax, sys; sys.exit(jax.devices()[0].platform not in ('gpu', 'cuda'))"
if subprocess.run([sys.executable, "-c", gpu_check]).returncode != 0:
    pip_install("-U", "jax[cuda12]")

import jax
import jax.numpy as jnp
from jax import lax
from huggingface_hub import hf_hub_download, login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF"))
except Exception:
    pass

T0 = time.time()
random.seed(0)
np.random.seed(0)
tmap = jax.tree_util.tree_map
print("devices:", jax.devices())

## 2. Hyperparameters and time budget

CONTEXT_TOKENS = 512
SENT_TOKENS = 128
MAX_SENTS = 48
TRAIN_TOKENS = 4096
PRED_TOKENS = 8192
LORMA_RANK = 16
LORMA_ALPHA = 16
LORMA_SCALE = LORMA_ALPHA / LORMA_RANK
KD_TEMP = 2.0
KD_ALPHA = 0.5
MAX_EPOCHS = 30
PATIENCE = 6
WARMUP_STEPS = 100
BIG = 1 << 30
EOS = 3
MAX_SECONDS = 7.5 * 3600
LORMA_MINUTES = 90
SENTENCE_MINUTES = 60
KD_MINUTES = 90


def budget(minutes):
    return min(minutes * 60, MAX_SECONDS - (time.time() - T0) - 2400)

devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 3. Load the Phase-1 checkpoint and tokenizer

In [2]:
CK_DIR = None

found = glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)
print("checkpoints found:", found)
CK_DIR = CK_DIR or os.path.dirname(found[0])

ck = pickle.load(open(f"{CK_DIR}/final_params.pkl", "rb"))
P0 = tmap(jnp.asarray, ck["params"])
cfg = ck["cfg"]
print("variant:", ck["variant"], cfg)
assert cfg["norm"] == "rms" and cfg["mlp"] == "swiglu" and cfg["pos"] == "rope" and cfg["qknorm"]

tk = json.load(open(f"{CK_DIR}/bpe_legal.json"))

V, D = P0["emb"].shape
L = P0["layers"]["wq"].shape[0]
HD = 64
H = P0["layers"]["wq"].shape[2] // HD
KV = cfg["kvh"]
R = H // KV
WIN = np.array([256 if (cfg["hybrid"] and i % 3 != 2) else BIG for i in range(L)], np.int32)
print(f"vocab {V}, dim {D}, layers {L}, heads {H}, kv heads {KV}, local windows {sorted(set(WIN.tolist()))}")

checkpoints found: ['/kaggle/input/datasets/nirbhaykachhatiya/phase3m1/final_params.pkl']
variant: ours_hybrid_gate {'norm': 'rms', 'mlp': 'swiglu', 'pos': 'rope', 'kvh': 4, 'qknorm': True, 'hybrid': True, 'gate': True, 'ff': 2048}
vocab 16384, dim 768, layers 12, heads 12, kv heads 4, local windows [256, 1073741824]


## 4. Sentence tokenizer (byte-level BPE from Phase 1)

In [3]:
PAT = regex.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+""")

byte_list = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
char_list = byte_list[:]
extra = 0
for b in range(256):
    if b not in byte_list:
        byte_list.append(b)
        char_list.append(256 + extra)
        extra += 1
B2U = {b: chr(c) for b, c in zip(byte_list, char_list)}

stoi = {s: i for i, s in enumerate(tk["itos"])}
rank = {tuple(m): i for i, m in enumerate(tk["merges"])}
bpe_cache = {}


def bpe(word):
    if word not in bpe_cache:
        syms = [B2U[b] for b in word.encode()]
        while len(syms) > 1:
            best_rank, best_i = min((rank.get(p, BIG), i) for i, p in enumerate(zip(syms, syms[1:])))
            if best_rank == BIG:
                break
            syms[best_i:best_i + 2] = [syms[best_i] + syms[best_i + 1]]
        bpe_cache[word] = [stoi[s] for s in syms]
    return bpe_cache[word]


def encode_sentence(sentence):
    ids = []
    for w in PAT.findall(" " + " ".join(str(sentence).split())):
        ids.extend(bpe(w))
    return ids[:SENT_TOKENS - 1] + [EOS]

## 5. Load Phase-3 data (CL and IT train / dev / test) and the label vocabulary

In [4]:
def hf_file(name):
    return hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{name}", repo_type="dataset", local_dir="/tmp/hf")


def load_docs(name):
    data = json.load(open(hf_file(name)))
    if isinstance(data, dict):
        data = [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    return data


def parse_labels(raw):
    if isinstance(raw, list):
        return [str(x) for x in raw]
    if all(str(k).isdigit() for k in raw):
        return [raw[k] for k in sorted(raw, key=int)]
    if all(isinstance(v, int) for v in raw.values()):
        return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    return list(raw)


LABELS = parse_labels(json.load(open(hf_file("Train/label_vocab.json"))))
K = len(LABELS)
LAB_ID = {l: i for i, l in enumerate(LABELS)}
print(K, "labels:", LABELS)

DOCS = {}
for dom in ("CL", "IT"):
    DOCS[dom] = {}
    for split in ("train", "dev", "test"):
        folder = "Test" if split == "test" else "Train"
        raw = load_docs(f"{folder}/{dom}_{split}.json")
        DOCS[dom][split] = [
            {
                "id": str(d["id"]),
                "sent_ids": [encode_sentence(s) for s in d["text"]],
                "y": np.array([LAB_ID[l] for l in d["labels"]], np.int32) if "labels" in d else None,
            }
            for d in raw
        ]
        n_sent = sum(len(d["sent_ids"]) for d in DOCS[dom][split])
        n_tok = sum(len(s) for d in DOCS[dom][split] for s in d["sent_ids"])
        print(f"{dom} {split}: {len(DOCS[dom][split])} docs, {n_sent} sentences, {n_tok / max(n_sent, 1):.1f} tokens/sentence")

for dom in ("CL", "IT"):
    y = np.concatenate([d["y"] for d in DOCS[dom]["train"]])
    print(dom, "train label counts:", dict(zip(LABELS, np.bincount(y, minlength=K).tolist())))

ALL_TRAIN = DOCS["CL"]["train"] + DOCS["IT"]["train"]
counts = np.bincount(np.concatenate([d["y"] for d in ALL_TRAIN]), minlength=K).astype(np.float64)
CLASS_W = np.where(counts > 0, np.clip((np.median(counts[counts > 0]) / np.maximum(counts, 1)) ** 0.5, 0.25, 4.0), 1.0).astype(np.float32)
print("class weights:", dict(zip(LABELS, CLASS_W.round(2).tolist())))

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

13 labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL train: 40 docs, 10593 sentences, 41.4 tokens/sentence


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

CL dev: 5 docs, 1396 sentences, 43.3 tokens/sentence


... (4 outputs left out)

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT dev: 5 docs, 746 sentences, 40.0 tokens/sentence


IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

IT test: 5 docs, 858 sentences, 44.3 tokens/sentence
CL train label counts: {'Fact': 2979, 'Issue': 17, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Statute': 367, 'Dissent': 284, 'PrecedentReliedUpon': 1305, 'PrecedentNotReliedUpon': 209, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 438, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'None': 53}
IT train label counts: {'Fact': 1119, 'Issue': 90, 'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Statute': 732, 'Dissent': 0, 'PrecedentReliedUpon': 640, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 436, 'RatioOfTheDecision': 1445, 'RulingByPresentCourt': 194, 'None': 244}
class weights: {'Fact': 0.46000000834465027, 'Issue': 2.859999895095825, 'ArgumentPetitioner': 0.7400000095367432, 'ArgumentRespondent': 0.8100000023841858, 'Statute': 0.8899999856948853, 'Dissent': 1.75, 'PrecedentReliedUpon': 0.6700000166893005, 'PrecedentNotReliedUpon': 1.2400000095367432, 'PrecedentOverruled

## 6. Sentence-only and context-aware windows

In [5]:
def make_windows(sent_ids, mode):
    n = len(sent_ids)
    if mode == "sentence":
        return [(i, i + 1) for i in range(n)]
    out, s = [], 0
    while s < n:
        e, tok = s, 0
        while e < n and e - s < MAX_SENTS and tok + len(sent_ids[e]) <= CONTEXT_TOKENS:
            tok += len(sent_ids[e])
            e += 1
        e = max(e, s + 1)
        out.append((s, e))
        if e >= n:
            break
        s += max(1, (e - s) // 2)
    return out


def build_set(docs, mode):
    T = SENT_TOKENS if mode == "sentence" else CONTEXT_TOKENS
    S = 1 if mode == "sentence" else MAX_SENTS
    ids, sidx, ys, meta = [], [], [], []
    for di, d in enumerate(docs):
        for s, e in make_windows(d["sent_ids"], mode):
            row = np.zeros(T, np.int32)
            si = np.full(T, -1, np.int32)
            y = np.full(S, -1, np.int32)
            pos = 0
            for j in range(s, e):
                seg = d["sent_ids"][j]
                row[pos:pos + len(seg)] = seg
                si[pos:pos + len(seg)] = j - s
                pos += len(seg)
                if d["y"] is not None:
                    y[j - s] = d["y"][j]
            ids.append(row)
            sidx.append(si)
            ys.append(y)
            meta.append((di, s, e))
    return {"ids": np.stack(ids), "sidx": np.stack(sidx), "y": np.stack(ys), "meta": meta}


SETS = {}


def get_sets(mode):
    if mode not in SETS:
        SETS[mode] = {
            "train": build_set(ALL_TRAIN, mode),
            "dev": {dom: build_set(DOCS[dom]["dev"], mode) for dom in ("CL", "IT")},
            "test": {dom: build_set(DOCS[dom]["test"], mode) for dom in ("CL", "IT")},
        }
        print(mode, "windows: train", len(SETS[mode]["train"]["ids"]), "| dev",
              {d: len(v["ids"]) for d, v in SETS[mode]["dev"].items()})
    return SETS[mode]


for m in ("context", "sentence"):
    get_sets(m)

context windows: train 3000 | dev {'CL': 258, 'IT': 126}
sentence windows: train 16845 | dev {'CL': 1396, 'IT': 746}


## 7. Model: Phase-1 backbone (causal) + LoRMA+ adapters + bidirectional head + sentence pooling

In [6]:
inv_freq = 1 / (10000 ** (np.arange(0, HD, 2) / HD))
angles = np.arange(1280)[:, None] * inv_freq
COS = jnp.asarray(np.cos(angles), jnp.float32)
SIN = jnp.asarray(np.sin(angles), jnp.float32)
CLASS_W_J = jnp.asarray(CLASS_W)
LOGIT_SENTS = {SENT_TOKENS: 1, CONTEXT_TOKENS: MAX_SENTS}
ADAPTED = ("wq", "wk", "wv", "wo", "w1", "w2", "w3")


def rmsn(x, w):
    return x * lax.rsqrt((x * x).mean(-1, keepdims=True) + 1e-6) * w


def rope(x, cos, sin):
    x1, x2 = x[..., :HD // 2], x[..., HD // 2:]
    return jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1)


def lin(p, name, x):
    y = x @ p[name]
    if name + "_mb" in p:
        y = y + LORMA_SCALE * (y @ p[name + "_ma"]) @ p[name + "_mb"]
    return y


def block(p, x, window, cos, sin, causal, valid=None):
    Bsz, T, _ = x.shape
    h = rmsn(x, p["n1"])
    q = rope(rmsn(lin(p, "wq", h).reshape(Bsz, T, H, HD), p["qn"]), cos, sin)
    k = rope(rmsn(lin(p, "wk", h).reshape(Bsz, T, KV, HD), p["kn"]), cos, sin)
    v = lin(p, "wv", h).reshape(Bsz, T, KV, HD)

    scores = jnp.einsum("btgrd,bsgd->bgrts", q.reshape(Bsz, T, KV, R, HD), k) * HD ** -0.5
    dist = jnp.arange(T)[:, None] - jnp.arange(T)[None]
    mask = ((dist >= 0) & (dist < window)) if causal else (jnp.abs(dist) < window)
    mask = mask[None] & (valid[:, None, :] if valid is not None else True)

    attn = jax.nn.softmax(jnp.where(mask[:, None, None], scores, -1e30), -1)
    out = lin(p, "wo", jnp.einsum("bgrts,bsgd->btgrd", attn, v).reshape(Bsz, T, H * HD))

    g1, g2 = (p["g"][0], p["g"][1]) if "g" in p else (1.0, 1.0)
    x = x + g1 * out
    h = rmsn(x, p["n2"])
    return x + g2 * lin(p, "w2", jax.nn.silu(lin(p, "w1", h)) * lin(p, "w3", h))


def logits_fn(P, win, ids, sidx):
    T = ids.shape[1]
    n_sent = LOGIT_SENTS[T]
    cos, sin = COS[:T][None, :, None, :], SIN[:T][None, :, None, :]
    valid = sidx >= 0

    def layer_step(x, z):
        return block(z[0], x, z[1], cos, sin, True), None

    x, _ = lax.scan(layer_step, P["emb"][ids], (P["layers"], win))
    x = block(P["head"], rmsn(x, P["nf"]), BIG, cos, sin, False, valid)

    onehot = jax.nn.one_hot(sidx, n_sent)
    pooled = jnp.einsum("bts,btd->bsd", onehot, x) / jnp.maximum(onehot.sum(1), 1.0)[..., None]
    return rmsn(pooled, P["nh"]) @ P["cls"] + P["cb"]


def init_head(key):
    ks = jax.random.split(key, 8)
    n = lambda i, shape: jax.random.normal(ks[i], shape) * 0.02
    ff = cfg["ff"]
    return {
        "wq": n(0, (D, H * HD)), "wk": n(1, (D, KV * HD)), "wv": n(2, (D, KV * HD)), "wo": n(3, (H * HD, D)),
        "n1": jnp.ones(D), "n2": jnp.ones(D), "qn": jnp.ones(HD), "kn": jnp.ones(HD),
        "w1": n(4, (D, ff)), "w3": n(5, (D, ff)), "w2": n(6, (ff, D)),
    }


def build_model(layers, key, lorma=False):
    layers = dict(layers)
    if lorma:
        for i, name in enumerate(ADAPTED):
            n_layers, d_in, d_out = layers[name].shape
            layers[name + "_ma"] = jax.random.normal(jax.random.fold_in(key, i), (n_layers, d_out, LORMA_RANK)) * 0.02
            layers[name + "_mb"] = jnp.zeros((n_layers, LORMA_RANK, d_out))
    return {
        "emb": P0["emb"],
        "layers": layers,
        "nf": P0["nf"],
        "head": init_head(jax.random.fold_in(key, 100)),
        "nh": jnp.ones(D),
        "cls": jax.random.normal(jax.random.fold_in(key, 101), (D, K)) * 0.02,
        "cb": jnp.zeros(K),
    }


def get_gates(P, method):
    def gate(path, x):
        if method == "full":
            return 1.0
        top, leaf = path[0].key, str(path[-1].key)
        return 1.0 if (top in ("head", "nh", "cls", "cb") or leaf.endswith(("_ma", "_mb"))) else 0.0
    return jax.tree_util.tree_map_with_path(gate, P)


def get_lr_mult(P, gates, head_mult):
    return jax.tree_util.tree_map_with_path(
        lambda path, x, g: g * (head_mult if path[0].key in ("head", "nh", "cls", "cb") else 1.0), P, gates)


def count_params(P, gates=None):
    leaves = jax.tree_util.tree_leaves(P)
    if gates is None:
        return sum(int(np.prod(x.shape)) for x in leaves)
    return sum(int(np.prod(x.shape)) for x, g in zip(leaves, jax.tree_util.tree_leaves(gates)) if g > 0)


logp_fn = jax.jit(lambda P, win, ids, sidx: jax.nn.log_softmax(logits_fn(P, win, ids, sidx)))
logits_jit = jax.jit(logits_fn)

## 8. Prediction (one label per sentence) and Macro-F1 / Precision / Recall

In [7]:
def predict_scores(P, win, data, docs):
    T = data["ids"].shape[1]
    pb = PRED_TOKENS // T
    acc = [np.zeros((len(d["sent_ids"]), K), np.float32) for d in docs]
    cnt = [np.zeros(len(d["sent_ids"]), np.float32) for d in docs]
    for i in range(0, len(data["ids"]), pb):
        ids, sidx = data["ids"][i:i + pb], data["sidx"][i:i + pb]
        pad = pb - len(ids)
        if pad:
            ids = np.concatenate([ids, np.zeros((pad, T), np.int32)])
            sidx = np.concatenate([sidx, np.full((pad, T), -1, np.int32)])
        lp = np.asarray(logp_fn(P, win, ids, sidx))
        for j, (di, s, e) in enumerate(data["meta"][i:i + pb]):
            acc[di][s:e] += lp[j, :e - s]
            cnt[di][s:e] += 1
    return [a / c[:, None] for a, c in zip(acc, cnt)]


def macro_scores(gold, pred):
    present = sorted(set(gold.tolist()) | set(pred.tolist()))
    f1_all = np.zeros(K)
    precs, recs, f1s = [], [], []
    for k in present:
        tp = int(((gold == k) & (pred == k)).sum())
        fp = int(((gold != k) & (pred == k)).sum())
        fn = int(((gold == k) & (pred != k)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        precs.append(pr)
        recs.append(rc)
        f1s.append(f1)
        f1_all[k] = f1
    return float(np.mean(f1s)), float(np.mean(precs)), float(np.mean(recs)), f1_all


def evaluate(P, win, mode):
    sets = get_sets(mode)
    res, golds, preds = {}, [], []
    for dom in ("CL", "IT"):
        scores = predict_scores(P, win, sets["dev"][dom], DOCS[dom]["dev"])
        pred = np.concatenate([s.argmax(1) for s in scores])
        gold = np.concatenate([d["y"] for d in DOCS[dom]["dev"]])
        res[dom] = macro_scores(gold, pred)
        golds.append(gold)
        preds.append(pred)
    pooled = macro_scores(np.concatenate(golds), np.concatenate(preds))
    return (res["CL"][0] + res["IT"][0]) / 2, res, pooled


def save_submission(name, P, win, mode):
    t_start = time.time()
    sets = get_sets(mode)
    result = {}
    for dom in ("CL", "IT"):
        scores = predict_scores(P, win, sets["test"][dom], DOCS[dom]["test"])
        for d, s in zip(DOCS[dom]["test"], scores):
            assert len(s) == len(d["sent_ids"])
            if d["id"] in result:
                print("warning: duplicate document id across domains:", d["id"])
            result[d["id"]] = [LABELS[i] for i in s.argmax(1)]
    elapsed = time.time() - t_start
    out_dir = f"/kaggle/working/{name}"
    os.makedirs(out_dir, exist_ok=True)
    json.dump(result, open(f"{out_dir}/prediction.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/{name}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{out_dir}/prediction.json", "prediction.json")
    print(f"{name}: {len(result)} documents, {sum(len(v) for v in result.values())} sentences, inference {elapsed:.0f}s")
    return elapsed

## 9. Training loop (class-weighted CE, optional distillation loss, gates via stop-gradient, early stopping)

In [8]:
def make_step(gates, lr_mult):
    def apply_gate(p, g):
        return p if g > 0 else lax.stop_gradient(p)

    def loss_fn(P, win, ids, sidx, y, teacher_logits, alpha):
        logits = logits_fn(tmap(apply_gate, P, gates), win, ids, sidx)
        logp = jax.nn.log_softmax(logits)
        keep = y >= 0
        yy = jnp.maximum(y, 0)
        w = CLASS_W_J[yy] * keep
        ce = (-jnp.take_along_axis(logp, yy[..., None], -1)[..., 0] * w).sum() / w.sum()

        t_logp = jax.nn.log_softmax(teacher_logits / KD_TEMP)
        s_logp = jax.nn.log_softmax(logits / KD_TEMP)
        kd = ((jnp.exp(t_logp) * (t_logp - s_logp)).sum(-1) * keep).sum() / jnp.maximum(keep.sum(), 1) * KD_TEMP ** 2
        return (1 - alpha) * ce + alpha * kd

    @jax.jit
    def step(P, m, v, t, lr, win, ids, sidx, y, teacher_logits, alpha):
        loss, g = jax.value_and_grad(loss_fn)(P, win, ids, sidx, y, teacher_logits, alpha)
        gnorm = jnp.sqrt(sum((x ** 2).sum() for x in jax.tree_util.tree_leaves(g)))
        g = tmap(lambda x: x * jnp.minimum(1.0, 1.0 / (gnorm + 1e-6)), g)
        m = tmap(lambda a, b: 0.9 * a + 0.1 * b, m, g)
        v = tmap(lambda a, b: 0.999 * a + 0.001 * b * b, v, g)

        def update(p, a, b, mult):
            adam = (a / (1 - 0.9 ** t)) / (jnp.sqrt(b / (1 - 0.999 ** t)) + 1e-8)
            return p - lr * mult * (adam + 0.01 * p)

        return tmap(update, P, m, v, lr_mult), m, v, loss

    return step


def train(name, P, win, mode, method, minutes, peak_lr, head_mult, teacher_logits=None, alpha=0.0):
    gates = get_gates(P, method)
    step = make_step(gates, get_lr_mult(P, gates, head_mult))
    m = tmap(jnp.zeros_like, P)
    v = tmap(jnp.zeros_like, P)
    data = get_sets(mode)["train"]
    T = data["ids"].shape[1]
    batch = TRAIN_TOKENS // T
    S = data["y"].shape[1]
    zeros = jnp.zeros((batch, S, K))
    steps_per_epoch = max(len(data["ids"]) // batch, 1)
    seconds = budget(minutes)
    print(f"[{name}] mode {mode} | trainable {count_params(P, gates) / 1e6:.2f}M of {count_params(P) / 1e6:.1f}M | {steps_per_epoch} steps/epoch")

    t, total, done, bad = 0, None, False, 0
    best = (-1, None, None)
    t_start = time.time()

    while not done:
        perm = np.random.permutation(len(data["ids"]))
        for i in range(steps_per_epoch):
            ix = perm[i * batch:(i + 1) * batch]
            t += 1
            lr = peak_lr * min(1, t / WARMUP_STEPS)
            if total:
                lr *= 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1, t / total)))
            tl = jnp.asarray(teacher_logits[ix], jnp.float32) if teacher_logits is not None else zeros
            P, m, v, loss = step(P, m, v, jnp.float32(t), lr, win, data["ids"][ix], data["sidx"][ix], data["y"][ix], tl, jnp.float32(alpha))
            if t == 5:
                loss.block_until_ready()
                t_bench = time.time()
            if t == 25:
                loss.block_until_ready()
                sec_per_step = (time.time() - t_bench) / 20
                total = min(MAX_EPOCHS * steps_per_epoch, int(0.9 * seconds / sec_per_step))
                print(f"[{name}] {sec_per_step:.2f}s/step -> {total} steps ({total / steps_per_epoch:.1f} epochs)")
            if (total and t >= total) or time.time() - T0 > MAX_SECONDS:
                done = True
                break

        mean_f1, res, pooled = evaluate(P, win, mode)
        print(f"[{name}] step {t} loss {float(loss):.4f} | dev macro-F1 CL {res['CL'][0]:.4f} IT {res['IT'][0]:.4f} mean {mean_f1:.4f} "
              f"| {(time.time() - t_start) / 60:.1f} min", flush=True)
        if mean_f1 > best[0]:
            best, bad = (mean_f1, P, (res, pooled)), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] early stopping")
                done = True

    return best, time.time() - t_start

## 10. Experiment 1 — LoRMA+ (rank 16), context-aware windows

In [9]:
RESULTS = []
WIN_FULL = jnp.asarray(WIN)

model = build_model(P0["layers"], jax.random.PRNGKey(0), lorma=True)
n_trainable = count_params(model, get_gates(model, "peft"))
(best_f1, teacher, (res, pooled)), train_secs = train("lorma_ctx", model, WIN_FULL, "context", "peft", LORMA_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_lorma", teacher, WIN_FULL, "context")
RESULTS.append(("lorma_ctx", n_trainable, count_params(teacher), best_f1, res, pooled, train_secs, infer_secs))

[lorma_ctx] mode context | trainable 8.96M of 97.1M | 375 steps/epoch
[lorma_ctx] 0.71s/step -> 6797 steps (18.1 epochs)
[lorma_ctx] step 375 loss 0.7942 | dev macro-F1 CL 0.3786 IT 0.5531 mean 0.4659 | 5.2 min
[lorma_ctx] step 750 loss 0.8738 | dev macro-F1 CL 0.3988 IT 0.5188 mean 0.4588 | 10.3 min
[lorma_ctx] step 1125 loss 0.2400 | dev macro-F1 CL 0.4070 IT 0.4955 mean 0.4512 | 15.3 min
[lorma_ctx] step 1500 loss 0.0383 | dev macro-F1 CL 0.4341 IT 0.5156 mean 0.4749 | 20.4 min
[lorma_ctx] step 1875 loss 0.1253 | dev macro-F1 CL 0.4812 IT 0.5118 mean 0.4965 | 25.5 min
[lorma_ctx] step 2250 loss 0.0246 | dev macro-F1 CL 0.4174 IT 0.5406 mean 0.4790 | 30.5 min
[lorma_ctx] step 2625 loss 0.1911 | dev macro-F1 CL 0.4999 IT 0.5265 mean 0.5132 | 35.6 min
[lorma_ctx] step 3000 loss 0.0005 | dev macro-F1 CL 0.4540 IT 0.5569 mean 0.5054 | 40.6 min
[lorma_ctx] step 3375 loss 0.0033 | dev macro-F1 CL 0.4149 IT 0.5229 mean 0.4689 | 45.7 min
[lorma_ctx] step 3750 loss 0.0005 | dev macro-F1 CL 0.

## 11. Experiment 2 — LoRMA+ (rank 16), sentence-only input (context ablation)

In [10]:
model = build_model(P0["layers"], jax.random.PRNGKey(0), lorma=True)
n_trainable = count_params(model, get_gates(model, "peft"))
(best_f1, best_P, (res, pooled)), train_secs = train("lorma_sent", model, WIN_FULL, "sentence", "peft", SENTENCE_MINUTES, 5e-4, 2.0)
infer_secs = save_submission("p3_lorma_sentence", best_P, WIN_FULL, "sentence")
RESULTS.append(("lorma_sent", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))
del best_P

[lorma_sent] mode sentence | trainable 8.96M of 97.1M | 526 steps/epoch
[lorma_sent] 0.57s/step -> 5693 steps (10.8 epochs)
[lorma_sent] step 526 loss 2.3031 | dev macro-F1 CL 0.2592 IT 0.3483 mean 0.3037 | 5.4 min
[lorma_sent] step 1052 loss 1.3700 | dev macro-F1 CL 0.2346 IT 0.4096 mean 0.3221 | 10.7 min
[lorma_sent] step 1578 loss 0.9034 | dev macro-F1 CL 0.2897 IT 0.3959 mean 0.3428 | 15.9 min
[lorma_sent] step 2104 loss 0.7988 | dev macro-F1 CL 0.2630 IT 0.3980 mean 0.3305 | 21.2 min
[lorma_sent] step 2630 loss 0.2478 | dev macro-F1 CL 0.2466 IT 0.3945 mean 0.3206 | 26.4 min
[lorma_sent] step 3156 loss 0.0958 | dev macro-F1 CL 0.2774 IT 0.3960 mean 0.3367 | 31.7 min
[lorma_sent] step 3682 loss 0.0247 | dev macro-F1 CL 0.2417 IT 0.3897 mean 0.3157 | 36.9 min
[lorma_sent] step 4208 loss 0.0154 | dev macro-F1 CL 0.2845 IT 0.3745 mean 0.3295 | 42.2 min
[lorma_sent] step 4734 loss 0.0007 | dev macro-F1 CL 0.2563 IT 0.4021 mean 0.3292 | 47.4 min
[lorma_sent] early stopping
p3_lorma_sent

## 12. Experiment 3 — Knowledge distillation (6-layer student, teacher = Experiment 1)

In [11]:
data = get_sets("context")["train"]
teacher_logits = np.concatenate([
    np.asarray(logits_jit(teacher, WIN_FULL, data["ids"][i:i + 16], data["sidx"][i:i + 16]))
    for i in range(0, len(data["ids"]), 16)
]).astype(np.float16)
print("teacher logits:", teacher_logits.shape)

student_idx = np.arange(0, L, 2)
WIN_STUDENT = jnp.asarray(WIN[student_idx])
student = build_model(tmap(lambda a: a[student_idx], P0["layers"]), jax.random.PRNGKey(1))
n_trainable = count_params(student)
(best_f1, best_P, (res, pooled)), train_secs = train(
    "kd_student", student, WIN_STUDENT, "context", "full", KD_MINUTES, 5e-5, 10.0, teacher_logits=teacher_logits, alpha=KD_ALPHA)
infer_secs = save_submission("p3_kd", best_P, WIN_STUDENT, "context")
RESULTS.append(("kd_student", n_trainable, count_params(best_P), best_f1, res, pooled, train_secs, infer_secs))

teacher logits: (3000, 48, 13)
[kd_student] mode context | trainable 56.65M of 56.6M | 375 steps/epoch
[kd_student] 0.46s/step -> 10562 steps (28.2 epochs)
[kd_student] step 375 loss 0.9471 | dev macro-F1 CL 0.3665 IT 0.4434 mean 0.4049 | 3.1 min
[kd_student] step 750 loss 0.4543 | dev macro-F1 CL 0.4211 IT 0.4690 mean 0.4451 | 6.1 min
[kd_student] step 1125 loss 0.1150 | dev macro-F1 CL 0.3941 IT 0.5144 mean 0.4542 | 9.1 min
[kd_student] step 1500 loss 0.2395 | dev macro-F1 CL 0.4073 IT 0.4839 mean 0.4456 | 12.1 min
[kd_student] step 1875 loss 0.1247 | dev macro-F1 CL 0.4425 IT 0.5088 mean 0.4757 | 15.1 min
[kd_student] step 2250 loss 0.0638 | dev macro-F1 CL 0.4172 IT 0.5364 mean 0.4768 | 18.2 min
[kd_student] step 2625 loss 0.1108 | dev macro-F1 CL 0.4388 IT 0.5066 mean 0.4727 | 21.2 min
[kd_student] step 3000 loss 0.0569 | dev macro-F1 CL 0.4129 IT 0.5107 mean 0.4618 | 24.2 min
[kd_student] step 3375 loss 0.1959 | dev macro-F1 CL 0.3995 IT 0.4794 mean 0.4395 | 27.2 min
[kd_student]

## 13. Results summary

In [12]:
print(f"{'experiment':12s}{'trainable(M)':>13s}{'total(M)':>10s}{'CL F1':>8s}{'CL P':>8s}{'CL R':>8s}{'IT F1':>8s}{'IT P':>8s}{'IT R':>8s}{'mean F1':>9s}{'train(min)':>11s}{'infer(s)':>9s}")
for name, n_tr, n_tot, mean_f1, res, pooled, tr, inf in RESULTS:
    cl, it = res["CL"], res["IT"]
    print(f"{name:12s}{n_tr / 1e6:13.2f}{n_tot / 1e6:10.1f}{cl[0]:8.4f}{cl[1]:8.4f}{cl[2]:8.4f}{it[0]:8.4f}{it[1]:8.4f}{it[2]:8.4f}{mean_f1:9.4f}{tr / 60:11.1f}{inf:9.0f}")

print()
print(f"{'label (pooled F1)':26s}" + "".join(f"{r[0]:>12s}" for r in RESULTS))
for k, label in enumerate(LABELS):
    print(f"{label:26s}" + "".join(f"{r[5][3][k]:12.4f}" for r in RESULTS))

ctx = next(r for r in RESULTS if r[0] == "lorma_ctx")
sent = next(r for r in RESULTS if r[0] == "lorma_sent")
print(f"\ncontext effect (LoRMA+ mean dev macro-F1): context {ctx[3]:.4f} vs sentence-only {sent[3]:.4f} -> {ctx[3] - sent[3]:+.4f}")
print(f"total time {(time.time() - T0) / 3600:.2f} h")

experiment   trainable(M)  total(M)   CL F1    CL P    CL R   IT F1    IT P    IT R  mean F1 train(min) infer(s)
lorma_ctx            8.96      97.1  0.4999  0.5169  0.5469  0.5265  0.6436  0.4941   0.5132       66.0       13
lorma_sent           8.96      97.1  0.2897  0.3404  0.3165  0.3959  0.4719  0.3733   0.3428       47.4       15
kd_student          56.65      56.6  0.4172  0.4362  0.4261  0.5364  0.6744  0.4980   0.4768       36.2        7

label (pooled F1)            lorma_ctx  lorma_sent  kd_student
Fact                            0.6745      0.5407      0.7116
Issue                           0.7586      0.5833      0.5600
ArgumentPetitioner              0.5972      0.4236      0.5437
ArgumentRespondent              0.5665      0.1829      0.6259
Statute                         0.8050      0.7015      0.7931
Dissent                         0.0000      0.0000      0.0000
PrecedentReliedUpon             0.4775      0.3462      0.4907
PrecedentNotReliedUpon          0.2254     